### PIPs

In [ ]:
!pip install -q "unsloth[kaggle-new] @ git+https://github.com/unslothai/unsloth.git" 
!pip install -U -q git+https://github.com/huggingface/trl.git bitsandbytes peft qwen-vl-utils trackio
!pip install -q --no-deps transformers==4.56.2 
!pip install -q --no-deps trl==0.22.2 

### Dataset

In [ ]:
from datasets import load_dataset, Dataset, DatasetDict

# --- Define Your Sample Sizes ---
TRAIN_SAMPLE_SIZE = 1000
VAL_SAMPLE_SIZE = 200

# (For a larger run, you can increase these numbers)
# TRAIN_SAMPLE_SIZE = 5000
# VAL_SAMPLE_SIZE = 500

print(f"Loading {TRAIN_SAMPLE_SIZE} train and {VAL_SAMPLE_SIZE} validation samples in streaming mode...")

# 1. Load the 'train' split in streaming mode
streamed_train = load_dataset(
    "apoidea/pubtabnet-html", 
    split='train',
    streaming=True  # <-- This is the magic flag
)

# 2. Load the 'validation' split in streaming mode
streamed_val = load_dataset(
    "apoidea/pubtabnet-html", 
    split='validation',
    streaming=True
)

# 3. Take only the N samples you want from the stream
# This creates an 'IterableDataset'
train_subset_iter = streamed_train.take(TRAIN_SAMPLE_SIZE)
val_subset_iter = streamed_val.take(VAL_SAMPLE_SIZE)

# 4. (Crucial Step) Convert the streamed datasets into standard Datasets
# The SFTTrainer works best with a standard Dataset (with a __len__)
# This loads *only* your tiny subset into memory.
train_subset = Dataset.from_list(list(train_subset_iter))
val_subset = Dataset.from_list(list(val_subset_iter))

# 5. Combine them into the DatasetDict your trainer expects
dataset = DatasetDict({
    'train': train_subset,
    'validation': val_subset
})

# 6. Instruction Setup 
instruction = "Generate the HTML representation for this table image."

print("\n✅ Successfully sampled dataset subset downloading the full")

Conversation Setup


In [ ]:
from PIL import Image

def convert_to_conversation(sample):
    return {
        "images" : [sample["image"]],
        "messages" :  [
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": instruction},
                    {"type": "image", "image": sample["image"]}
                ],
            },
            {
                "role": "assistant",
                "content": [
                    {"type": "text", "text": sample["html_table"]}
            ],
            },
        ]
    }

pass

In [ ]:
# Training + Evaluation Set Initialization 
converted_dataset = [convert_to_conversation(sample) for sample in dataset['train']]
val_dataset = [convert_to_conversation(sample) for sample in dataset['validation']]

### Model Setup + Training Loop

In [ ]:
import torch
from unsloth import FastVisionModel
from datasets import load_dataset
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig

In [ ]:
import re
from IPython.display import display, HTML

def clean_html_output(text):
    
    # 1. Remove Everything Not Markdown
    pattern = r"```html\s*(.*?)\s*```"
    match = re.search(pattern, text, re.DOTALL)
    if match:
        text = match.group(1)
    
    # 2. Fallback: Remove generic code blocks
    text = text.replace("```", "")
    
    # 3. Strip leading/trailing whitespace
    return text.strip()

def generate_text_from_sample(
    model, 
    tokenizer, 
    image, 
    instruction="Generate the HTML representation for this table image.",
    max_new_tokens=1024,
    temperature=0.5,
    min_p=0.1,
    debug_display=True,
    device="None",
): 
    if device is None:
        device = model.device
        
    # 1. Enable inference mode optimization
    FastVisionModel.for_inference(model)
    # 2. Format the message for Qwen2.5-VL
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image"},
                {"type": "text", "text": instruction}
            ]
        }
    ]
    # 3. Apply the chat template
    input_text = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
    )
    # 4. Process inputs
    inputs = tokenizer(
        images=[image],
        text=[input_text],
        return_tensors="pt",
    ).to(device)
    # 5. Generate the output
    outputs = model.generate(
        **inputs, 
        max_new_tokens=max_new_tokens,
        use_cache=True,
        temperature=temperature, 
        min_p=min_p
    )
    # 6. Decode the output
    generated_text = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
    # 7. Clean the output
    # Basic Markdown Stripping
    final_output = clean_html_output(generated_text)
    # 8. Display results if requested
    if debug_display:
        # print("--- Generated Text ---")
        # print(generated_text)
        print("\n--- Rendered HTML Output ---")
        display(HTML(final_output))
        print("\n--- Original Image ---")
        display(image)

    return generated_text

In [ ]:
model, tokenizer = FastVisionModel.from_pretrained(
    "unsloth/Qwen2.5-VL-7B-Instruct-bnb-4bit",
    load_in_4bit = True,
    use_gradient_checkpointing = "unsloth",
    device_map = "balanced",
)
# Run the function
html_result = generate_text_from_sample(
    model=model,
    tokenizer=tokenizer,
    image=sample_image,
    device="cuda:0"
)

In [ ]:
# Cache Cleaner
import gc
import time
import torch

def clear_memory():
    # Delete variables if they exist in the current global scope
    if 'inputs' in globals(): del globals()['inputs']
    if 'model' in globals(): del globals()['model']
    if 'processor' in globals(): del globals()['processor']
    if 'trainer' in globals(): del globals()['trainer']
    if 'bnb_config' in globals(): del globals()['bnb_config']
    time.sleep(2)

    # Garbage collection and clearing CUDA memory
    gc.collect()
    time.sleep(2)
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    time.sleep(2)
    gc.collect()
    time.sleep(2)

    print(f"GPU allocated memory: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
    print(f"GPU reserved memory: {torch.cuda.memory_reserved() / 1024**3:.2f} GB")

clear_memory()

In [ ]:
from transformers import AutoProcessor

model, tokenizer = FastVisionModel.from_pretrained(
    "unsloth/Qwen2.5-VL-7B-Instruct-bnb-4bit",
    load_in_4bit = True,
    use_gradient_checkpointing = "unsloth",
)

model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers     = True,
    finetune_language_layers   = True,
    finetune_attention_modules = True,
    finetune_mlp_modules       = True,
    r = 16,
    lora_alpha = 32,
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

training_args =  SFTConfig(
        per_device_train_batch_size = 2, # effective maintained @8
        gradient_accumulation_steps = 4,
    
        # For 1k data, 3 epochs (375 steps) is a sweet spot for initial testing.
        num_train_epochs = 3,
    
        warmup_steps = 40, #10% of total steps
    
        # num_train_epochs = 1, # Set this instead of max_steps for full training runs
        learning_rate = 2e-4,
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "cosine",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",

        # You MUST put the below items for vision finetuning:
        remove_unused_columns = False,
        dataset_text_field = "",
        dataset_kwargs = {"skip_prepare_dataset": True},
        max_length = 1028,
    )

In [ ]:
FastVisionModel.for_training(model) # Enable for training!

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    data_collator = UnslothVisionDataCollator(model, tokenizer), # Must use!
    train_dataset = converted_dataset,
    eval_dataset = val_dataset,
    args = training_args,
    device = "cuda:1",
)

In [ ]:
trainer_stats = trainer.train()

In [ ]:
model.save_pretrained("lora_model")
tokenizer.save_pretrained("lora_model")

print("Training complete and LoRA model saved to 'lora_model' directory.")

In [ ]:
clear_memory()

In [ ]:
model, tokenizer = FastVisionModel.from_pretrained(
    "/kaggle/working/lora_model", 
    load_in_4bit = True,
    # use_gradient_checkpointing = "unsloth",
    device_map="balanced",
    
)
html_result = generate_text_from_sample(
    model = model,
    tokenizer = tokenizer,
    image = sample_image,
    device = "cuda:1",
)
print(html_result)